# MNIST Image Classification
Training MLP, CNN, and Transformer models on the MNIST dataset.

## Load MNIST Dataset

In [ ]:
import pandas as pd

splits = {'train': 'mnist/train-00000-of-00001.parquet', 'test': 'mnist/test-00000-of-00001.parquet'}
train = pd.read_parquet("hf://datasets/ylecun/mnist/" + splits["train"])
test = pd.read_parquet("hf://datasets/ylecun/mnist/" + splits["test"])

In [ ]:
from PIL import Image
import io
import numpy as np

# decode image bytes -> flat numpy array
def image_to_vector(image_dict):
    img = Image.open(io.BytesIO(image_dict['bytes']))
    return np.array(img).flatten()

train['image_vector'] = train['image'].apply(image_to_vector)
test['image_vector'] = test['image'].apply(image_to_vector)

In [ ]:
# normalize to [0, 1]
train['normalized_image_vector'] = train['image_vector'].apply(lambda x: x / 255.0)
test['normalized_image_vector'] = test['image_vector'].apply(lambda x: x / 255.0)

X_train, y_train = train['normalized_image_vector'], train['label'].to_numpy()
X_test, y_test = test['normalized_image_vector'], test['label'].to_numpy()
X_train = np.array(X_train.to_list())
X_test = np.array(X_test.to_list())

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 64
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_dataloader)}, Test batches: {len(test_dataloader)}")

## Multi-Layer Perceptron (MLP)

In [ ]:
import torch.nn as nn
import torch.optim as optim

class MLP(nn.Module):
    def __init__(self, hidden_size):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(784, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, 10)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

hidden_size = 128
num_epochs = 5
learning_rate = 0.001

model = MLP(hidden_size)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    model.train()
    for i, (X_batch, y_batch) in enumerate(train_dataloader):
        outputs = model(X_batch)
        loss = loss_fn(outputs, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if (i+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_dataloader)}], Loss: {loss.item():.4f}')

model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for X_batch, y_batch in test_dataloader:
        outputs = model(X_batch)
        _, y_pred = torch.max(outputs, 1)
        total += len(y_batch)
        correct += (y_pred == y_batch).sum().item()
    print(f'MLP Test Accuracy: {100 * correct / total:.2f}%')

## Convolutional Neural Network (CNN)

In [ ]:
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # 28x28 -> conv+pool -> 14x14 -> conv+pool -> 7x7
        self.conv1 = nn.Conv2d(1, 16, kernel_size=5, padding=2)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5, padding=2)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        out = self.pool1(F.relu(self.conv1(x)))
        out = self.pool2(F.relu(self.conv2(out)))
        out = out.reshape(out.size(0), -1)
        out = self.fc(out)
        return out

num_epochs = 5
learning_rate = 0.001

cnn_model = CNN()
loss_fn_cnn = nn.CrossEntropyLoss()
optimizer_cnn = optim.Adam(cnn_model.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    cnn_model.train()
    for i, (X_batch, y_batch) in enumerate(train_dataloader):
        outputs = cnn_model(X_batch.view(-1, 1, 28, 28))
        loss = loss_fn_cnn(outputs, y_batch)
        optimizer_cnn.zero_grad()
        loss.backward()
        optimizer_cnn.step()
        if (i+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_dataloader)}], Loss: {loss.item():.4f}')

cnn_model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for X_batch, y_batch in test_dataloader:
        outputs = cnn_model(X_batch.view(-1, 1, 28, 28))
        _, y_pred = torch.max(outputs, 1)
        total += len(y_batch)
        correct += (y_pred == y_batch).sum().item()
    print(f'CNN Test Accuracy: {100 * correct / total:.2f}%')

## Transformer (Encoder)

In [ ]:
import math

# split image into patches and project to embed_dim
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)        # (B, embed_dim, h, w)
        x = x.flatten(2)        # (B, embed_dim, num_patches)
        x = x.transpose(1, 2)  # (B, num_patches, embed_dim)
        return x


# sinusoidal positional encoding
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class ImageTransformer(nn.Module):
    def __init__(self, img_size=28, patch_size=7, in_channels=1, num_classes=10,
                 embed_dim=64, num_heads=4, num_layers=2, hidden_dim_mlp=128):
        super().__init__()
        self.patch_embedding = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        self.positional_encoding = PositionalEncoding(embed_dim, self.patch_embedding.num_patches)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=hidden_dim_mlp, dropout=0.1, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = x.view(-1, 1, self.patch_embedding.img_size, self.patch_embedding.img_size)
        x = self.patch_embedding(x)
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)           # global average pooling over patches
        return self.classifier(x)

In [ ]:
num_epochs = 5
learning_rate = 0.001

transformer_model = ImageTransformer(
    img_size=28, patch_size=7, in_channels=1, num_classes=10,
    embed_dim=64, num_heads=4, num_layers=2, hidden_dim_mlp=128
)

loss_fn_transformer = nn.CrossEntropyLoss()
optimizer_transformer = optim.Adam(transformer_model.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    transformer_model.train()
    for i, (X_batch, y_batch) in enumerate(train_dataloader):
        outputs = transformer_model(X_batch)
        loss = loss_fn_transformer(outputs, y_batch)
        optimizer_transformer.zero_grad()
        loss.backward()
        optimizer_transformer.step()
        if (i+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_dataloader)}], Loss: {loss.item():.4f}')

transformer_model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for X_batch, y_batch in test_dataloader:
        outputs = transformer_model(X_batch)
        _, y_pred = torch.max(outputs, 1)
        total += len(y_batch)
        correct += (y_pred == y_batch).sum().item()
    print(f'Transformer Test Accuracy: {100 * correct / total:.2f}%')

## Image Augmentation (bonus)
Reload data with `torchvision` to apply augmentations and see if they improve test accuracy.

In [ ]:
from torchvision import datasets, transforms

# augmentations on training set: random rotation + horizontal flip
train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_aug = datasets.MNIST(root='./data', train=True, download=True, transform=train_transform)
test_aug = datasets.MNIST(root='./data', train=False, download=True, transform=test_transform)

train_loader_aug = DataLoader(train_aug, batch_size=64, shuffle=True)
test_loader_aug = DataLoader(test_aug, batch_size=64, shuffle=False)

print(f"Augmented train batches: {len(train_loader_aug)}, test batches: {len(test_loader_aug)}")
imgs, labels = next(iter(train_loader_aug))
print(f"Batch shape: {imgs.shape}, Labels shape: {labels.shape}")

In [ ]:
# retrain CNN with augmented data
cnn_aug = CNN()
loss_fn_aug = nn.CrossEntropyLoss()
optimizer_aug = optim.Adam(cnn_aug.parameters(), lr=0.001)

for epoch in range(5):
    cnn_aug.train()
    for i, (X_batch, y_batch) in enumerate(train_loader_aug):
        outputs = cnn_aug(X_batch)
        loss = loss_fn_aug(outputs, y_batch)
        optimizer_aug.zero_grad()
        loss.backward()
        optimizer_aug.step()
        if (i+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/5], Step [{i+1}/{len(train_loader_aug)}], Loss: {loss.item():.4f}')

cnn_aug.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for X_batch, y_batch in test_loader_aug:
        _, y_pred = torch.max(cnn_aug(X_batch), 1)
        total += len(y_batch)
        correct += (y_pred == y_batch).sum().item()
    print(f'CNN + Augmentation Test Accuracy: {100 * correct / total:.2f}%')